# Dataset validation analysis

Validate the full-endpoint publication dataset through explicit UK Biobank mentions, keyword categories, model-agreement fields, semantic structure, discriminative terms, and publication-year trends.


In [ ]:
import sys
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

from utils import shared_paths as P
from utils.data_analysis_00_dataset_analysis import (
    CATEGORY_PATTERNS,
    MODEL_NAMES,
    contains_pattern,
    load_publications,
    model_agreement_columns,
    normalise_bool,
    normalized_rows,
    output_dirs,
    sample_balanced,
    save_figure,
)

P.bootstrap()


In [ ]:
df = load_publications(P.SHOWCASE_PLUS)
df.shape


## 1. Explicit UK Biobank mentions

Measure explicit UK Biobank mentions in publication titles and abstracts.


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_validation_analysis/01_explicit_ukb_mentions")

explicit_summary = pd.DataFrame(
    {
        "n_publications": [len(df)],
        "n_explicit_ukb_mentions": [int(df["explicit_ukb_mention"].sum())],
        "explicit_ukb_mention_percent": [float(df["explicit_ukb_mention"].mean() * 100)],
    }
)
explicit_summary.to_csv(table_dir / "explicit_ukb_mention_summary.csv", index=False)

explicit_yearly = (
    df.dropna(subset=["analysis_year"])
    .groupby("analysis_year")
    .agg(
        n_publications=("id", "count"),
        n_explicit_ukb_mentions=("explicit_ukb_mention", "sum"),
    )
    .reset_index()
)
explicit_yearly["explicit_ukb_mention_percent"] = (
    explicit_yearly["n_explicit_ukb_mentions"] / explicit_yearly["n_publications"] * 100
)
explicit_yearly.to_csv(table_dir / "explicit_ukb_mentions_by_year.csv", index=False)

figure, axis = plt.subplots(figsize=(11, 6))
axis.plot(explicit_yearly["analysis_year"], explicit_yearly["explicit_ukb_mention_percent"], marker="o")
axis.set(xlabel="Publication year", ylabel="Explicit UKB mention (%)", title="Explicit UK Biobank mentions by year")
axis.grid(alpha=0.25)
save_figure(figure, figure_dir / "explicit_ukb_mentions_by_year.png")
explicit_summary


## 2. Keyword categories

Profile keyword categories across the full publication corpus.


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_validation_analysis/02_keyword_categories")

category_rows = []
for category, pattern in CATEGORY_PATTERNS.items():
    hits = df["analysis_text"].apply(lambda text: contains_pattern(text, pattern))
    category_rows.append(
        {
            "category": category,
            "n_publications": len(df),
            "n_with_category": int(hits.sum()),
            "percent_with_category": float(hits.mean() * 100),
        }
    )

category_summary = pd.DataFrame(category_rows).sort_values("percent_with_category")
category_summary.to_csv(table_dir / "keyword_category_summary.csv", index=False)

figure, axis = plt.subplots(figsize=(10, 8))
axis.barh(category_summary["category"], category_summary["percent_with_category"])
axis.set(xlabel="Publications with cue (%)", title="Keyword profile of the full publication corpus")
axis.grid(axis="x", alpha=0.25)
save_figure(figure, figure_dir / "keyword_category_profile.png")
category_summary


## 3. Model agreement

Audit and summarize LLM agreement columns when they are present in the publication parquet.


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_validation_analysis/03_model_agreement")
required = model_agreement_columns()
available = sorted(required & set(df.columns))
missing = sorted(required - set(df.columns))

schema_audit = pd.DataFrame(
    {
        "metric": ["required_model_columns", "available_model_columns", "missing_model_columns", "analysis_ready"],
        "value": [len(required), len(available), len(missing), not missing],
        "detail": ["", ", ".join(available), ", ".join(missing), ""],
    }
)
schema_audit.to_csv(table_dir / "model_agreement_schema_audit.csv", index=False)
schema_audit


In [ ]:
if missing:
    print("Model-agreement calculations were not run because the publication endpoint parquet does not contain LLM prediction columns.")
else:
    agreement = df.copy()
    for model in MODEL_NAMES:
        for suffix in ("parse_ok", "true", "false"):
            agreement[f"{model}_{suffix}"] = normalise_bool(agreement[f"{model}_{suffix}"])
    for column in ("three_model_TRUE_agreement", "three_model_FALSE_agreement", "all_three_parsed"):
        agreement[column] = normalise_bool(agreement[column])

    model_rows = []
    for model in MODEL_NAMES:
        parsed = int(agreement[f"{model}_parse_ok"].sum())
        true_count = int(agreement[f"{model}_true"].sum())
        model_rows.append(
            {
                "model": model,
                "parsed": parsed,
                "true": true_count,
                "false": int(agreement[f"{model}_false"].sum()),
                "true_percent_among_parsed": true_count / max(parsed, 1) * 100,
            }
        )
    model_summary = pd.DataFrame(model_rows)
    model_summary.to_csv(table_dir / "model_positive_rate_summary.csv", index=False)

    vote_distribution = (
        agreement["n_true_votes"].value_counts().sort_index().rename_axis("n_true_votes").reset_index(name="n_candidates")
    )
    vote_distribution.to_csv(table_dir / "true_vote_distribution.csv", index=False)

    consensus_distribution = (
        agreement["consensus_group"].value_counts().rename_axis("consensus_group").reset_index(name="n_candidates")
    )
    consensus_distribution.to_csv(table_dir / "consensus_group_distribution.csv", index=False)

    figure, axis = plt.subplots(figsize=(8.5, 5))
    axis.bar(model_summary["model"], model_summary["true_percent_among_parsed"])
    axis.set(ylabel="TRUE among parsed rows (%)", title="Model-level positive rate")
    axis.grid(axis="y", alpha=0.25)
    save_figure(figure, figure_dir / "model_positive_rate_summary.png")
    display(model_summary)


## 4. Semantic map

Map semantic structure and compare publications with and without explicit UK Biobank mentions.


In [ ]:
MAX_PER_GROUP = 3000
RANDOM_STATE = 42
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

semantic_sample = sample_balanced(df, "explicit_ukb_mention", MAX_PER_GROUP, RANDOM_STATE)
semantic_texts = semantic_sample["analysis_text"].str.slice(0, 3500).tolist()

try:
    from sentence_transformers import SentenceTransformer

    embedding_model = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = embedding_model.encode(
        semantic_texts,
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    embedding_method = EMBEDDING_MODEL
except Exception:
    semantic_vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        max_features=12000,
    )
    semantic_matrix = semantic_vectorizer.fit_transform(semantic_texts)
    components = min(100, semantic_matrix.shape[0] - 1, semantic_matrix.shape[1] - 1)
    embeddings = normalized_rows(
        TruncatedSVD(n_components=components, random_state=RANDOM_STATE).fit_transform(semantic_matrix)
    )
    embedding_method = "TF-IDF + TruncatedSVD"

semantic_labels = semantic_sample["explicit_ukb_mention"].astype(int).to_numpy()
coordinates = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(embeddings)
semantic_sample["semantic_x"] = coordinates[:, 0]
semantic_sample["semantic_y"] = coordinates[:, 1]
silhouette = (
    silhouette_score(embeddings, semantic_labels, metric="cosine")
    if len(np.unique(semantic_labels)) > 1
    else np.nan
)


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_validation_analysis/04_semantic_map")
semantic_sample.to_csv(table_dir / "semantic_sample_with_coordinates.csv", index=False)
semantic_metrics = pd.DataFrame(
    [{"embedding_method": embedding_method, "sample_size": len(semantic_sample), "silhouette_index_cosine": silhouette}]
)
semantic_metrics.to_csv(table_dir / "semantic_metrics.csv", index=False)

figure, axis = plt.subplots(figsize=(9, 7))
for explicit, group in semantic_sample.groupby("explicit_ukb_mention"):
    label = "Explicit UKB mention" if explicit else "No explicit UKB mention"
    axis.scatter(group["semantic_x"], group["semantic_y"], s=12, alpha=0.55, label=f"{label} (n={len(group):,})")
axis.set(xlabel="Semantic component 1", ylabel="Semantic component 2", title=f"Semantic map (silhouette={silhouette:.4f})")
axis.grid(alpha=0.25)
axis.legend()
save_figure(figure, figure_dir / "semantic_map_explicit_ukb_mentions.png")
semantic_metrics


## 5. TF-IDF terms

Compare discriminative terms for publications with and without explicit UK Biobank mentions.


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_validation_analysis/05_tfidf_terms")
tfidf_sample = sample_balanced(df, "explicit_ukb_mention", 20000, 42)
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,
    max_features=12000,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(tfidf_sample["analysis_text"].fillna(""))
terms = np.asarray(tfidf_vectorizer.get_feature_names_out())
explicit_mask = tfidf_sample["explicit_ukb_mention"].to_numpy()
explicit_mean = np.asarray(tfidf_matrix[explicit_mask].mean(axis=0)).ravel()
other_mean = np.asarray(tfidf_matrix[~explicit_mask].mean(axis=0)).ravel()

term_summary = pd.DataFrame(
    {
        "term": terms,
        "mean_tfidf_explicit_ukb": explicit_mean,
        "mean_tfidf_no_explicit_ukb": other_mean,
        "difference_explicit_minus_other": explicit_mean - other_mean,
    }
).sort_values("difference_explicit_minus_other", ascending=False)
term_summary.to_csv(table_dir / "tfidf_discriminative_terms.csv", index=False)

figure, axes = plt.subplots(1, 2, figsize=(16, 9))
top_explicit = term_summary.head(25).iloc[::-1]
top_other = term_summary.tail(25).sort_values("difference_explicit_minus_other").iloc[::-1]
axes[0].barh(top_explicit["term"], top_explicit["difference_explicit_minus_other"])
axes[0].set(title="Associated with explicit UKB mentions", xlabel="Mean TF-IDF difference")
axes[1].barh(top_other["term"], -top_other["difference_explicit_minus_other"])
axes[1].set(title="Associated with no explicit UKB mention", xlabel="Mean TF-IDF difference")
for axis in axes:
    axis.grid(axis="x", alpha=0.25)
figure.tight_layout()
save_figure(figure, figure_dir / "tfidf_discriminative_terms.png")
term_summary.head(10)


## 6. Publication-year trends

Summarize yearly publication volume and explicit UK Biobank mentions.


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_validation_analysis/06_year_trends")
yearly_trends = explicit_yearly.copy()
yearly_trends["n_without_explicit_ukb_mention"] = (
    yearly_trends["n_publications"] - yearly_trends["n_explicit_ukb_mentions"]
)
yearly_trends.to_csv(table_dir / "publication_year_trends.csv", index=False)

figure, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
axes[0].plot(yearly_trends["analysis_year"], yearly_trends["n_publications"], marker="o")
axes[0].set(ylabel="Publications", title="UK Biobank publication corpus by year")
axes[1].plot(yearly_trends["analysis_year"], yearly_trends["explicit_ukb_mention_percent"], marker="o")
axes[1].set(xlabel="Publication year", ylabel="Explicit UKB mention (%)")
for axis in axes:
    axis.grid(alpha=0.25)
figure.tight_layout()
save_figure(figure, figure_dir / "publication_year_trends.png")
yearly_trends.tail(10)
